# Survival Analysis — Cox Proportional Hazards

Python translation of `Code/survival_analysis.R`.

In [16]:
import geopandas as gpd
import pandas as pd
import numpy as np
from lifelines import CoxPHFitter
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
SHP_PATH = PROJECT_ROOT / "Data" / "Processed" / "northParishFlows.shp"
OUT_TEX  = PROJECT_ROOT / "Output" / "Tables" / "python_survival.tex"
DAY = 40  # censoring day

## Load and process data

In [17]:
pdf = gpd.read_file(SHP_PATH)
rdf = pd.DataFrame(pdf.drop(columns="geometry"))

# Replace day < 1 with DAY; fill NAs with DAY
rdf["day"] = rdf["day"].where(rdf["day"] >= 1, DAY).fillna(DAY)

# primary: fill NA with 0 (no event observed within study period)
rdf["primary"] = rdf["primary"].fillna(0)

# primary_day: day * primary, then replace values < 1 with DAY
rdf["primary_day"] = rdf["day"] * rdf["primary"]
rdf["primary_day"] = rdf["primary_day"].where(rdf["primary_day"] >= 1, DAY)

# Survival times: days from news arrival to event or censoring
rdf["survival"]         = rdf["day"]         - rdf["news_day"]
rdf["primary_survival"] = rdf["primary_day"] - rdf["news_day"]
rdf["primary_survival"] = rdf["primary_survival"].fillna(DAY)

## Standardise continuous variables

In [18]:
# Matches R's scale(): centre=TRUE, scale=TRUE (sample std, ddof=1)
scale_vars = [
    "llo_arak", "lti_sk", "lal_sk", "lLStax_pc", "lpopC",
    "Y_COORD", "area", "mean_slope", "wet_1535", "wet_1536", "dwx_1536",
]
for v in scale_vars:
    rdf[v] = (rdf[v] - rdf[v].mean()) / rdf[v].std()

## Cox Proportional Hazards Models

In [19]:
BASE_COVARS = ["llo_arak", "lti_sk", "lal_sk", "smHouse", "bigHouse", "friary", "wet_1536"]
GEO_COVARS  = ["Y_COORD", "area", "uplands", "lowlands", "mean_slope"]
DURATION    = "primary_survival"
EVENT       = "primary"


def fit_cox(covars, df, penalizer=0.0):
    cols = covars + [DURATION, EVENT]
    data = df[cols].dropna()
    n_events = int(data[EVENT].sum())
    n_params = len(covars)
    if n_events <= n_params:
        print(
            f"Warning: events ({n_events}) <= parameters ({n_params}). "
            "Convergence may be unstable without penalization."
        )
    cph = CoxPHFitter(penalizer=penalizer)
    cph.fit(data, duration_col=DURATION, event_col=EVENT)
    cph.n_obs = len(data)  # store observation count for table output
    return cph


cox1 = fit_cox(BASE_COVARS, rdf)
cox2 = fit_cox(BASE_COVARS + ["lLStax_pc", "lpopC"], rdf)
# Mild ridge penalty stabilizes model 3's Hessian (20 events vs 14 predictors).
cox3 = fit_cox(BASE_COVARS + ["lLStax_pc", "lpopC"] + GEO_COVARS, rdf, penalizer=0.01)

In [20]:
# Quick display of model summaries
for i, m in enumerate([cox1, cox2, cox3], 1):
    print(f"\n=== Model {i} ===")
    print(m.summary[["coef", "se(coef)", "p"]].round(4))


=== Model 1 ===
             coef  se(coef)       p
covariate                          
llo_sk     0.2347    0.2420  0.3322
lti_sk    -0.0498    0.2108  0.8134
lal_sk     0.0034    0.1357  0.9798
smHouse    1.0543    0.5590  0.0593
bigHouse   2.0050    0.7914  0.0113
friary     0.3599    0.8365  0.6670
wet_1536   0.7969    0.2230  0.0004

=== Model 2 ===
             coef  se(coef)       p
covariate                          
llo_sk     0.0585    0.2900  0.8400
lti_sk    -0.2204    0.2663  0.4079
lal_sk     0.0286    0.1738  0.8693
smHouse    1.2693    0.5816  0.0291
bigHouse   1.8695    0.9552  0.0503
friary     0.2626    0.9918  0.7912
wet_1536   0.5633    0.2418  0.0198
lLStax_pc -0.0240    0.2677  0.9284
lpopC      0.4426    0.1364  0.0012

=== Model 3 ===
              coef  se(coef)       p
covariate                           
llo_sk      0.0727    0.1979  0.7135
lti_sk     -0.1023    0.1813  0.5726
lal_sk      0.0549    0.1509  0.7161
smHouse     0.9644    0.5207  0.0640
bigHous

## Export LaTeX table (stargazer-style)

In [ ]:
# --- Table configuration ---
# BS = chr(92) avoids backslash escape-sequence noise in the strings below
BS = chr(92)

DISPLAY_COVARS = BASE_COVARS + ["lLStax_pc", "lpopC"]
COVAR_LABELS = [
    f"ln(Land Owned / km{BS}textsuperscript{{2}})",
    f"ln(Tithe / km{BS}textsuperscript{{2}})",
    f"ln(Alms / km{BS}textsuperscript{{2}})",
    "Small House Dummy",
    "Large House Dummy",
    "Friary",
    "Wet 1535",
    "Wet 1536",
    "ln(1535 Lay Subsidy Amount)",
    "ln(Population)",
]
ADD_LINES = [
    ("Population",          ["N", "Y", "Y"]),
    ("Geographic Controls", ["N", "N", "Y"]),
]
COLUMN_LABELS = ["Land", "Taxation and Population", "Geographic Controls"]


def _stars(p):
    """Return LaTeX significance stars for a p-value."""
    if p < 0.01: return "$^{***}$"
    if p < 0.05: return "$^{**}$"
    if p < 0.10: return "$^{*}$"
    return ""


def cox_to_latex(
    models, title, display_covars, covar_labels,
    column_labels, add_lines, output_path, table_placement="H",
):
    """Write a stargazer-style LaTeX table for a list of CoxPHFitter models."""
    n      = len(models)
    hline  = BS + "hline"
    hhline = hline + hline
    newrow = " " + BS + BS  # LaTeX line-break token

    def row(*cells):
        return " & ".join(str(c) for c in cells) + newrow

    lines = []
    lines.append(BS + f"begin{{table}}[{table_placement}]")
    lines.append(BS + f"caption{{{title}}}")
    lines.append(BS + "label{tab:survival}")
    lines.append(BS + "centering")
    lines.append(BS + f"begin{{tabular}}{{l{'c' * n}}}")
    lines.append(hhline)

    # Column labels
    lines.append(row("", *[BS + f"textit{{{lbl}}}" for lbl in column_labels]))
    lines.append(row("", *[f"({i + 1})" for i in range(n)]))
    lines.append(hline)

    # Covariate rows: coefficient then standard error in parentheses
    for covar, label in zip(display_covars, covar_labels):
        coefs, ses = [], []
        for m in models:
            s = m.summary
            if covar in s.index:
                c  = s.loc[covar, "coef"]
                se = s.loc[covar, "se(coef)"]
                p  = s.loc[covar, "p"]
                coefs.append(f"{c:.3f}{_stars(p)}")
                ses.append(f"({se:.3f})")
            else:
                coefs.append("")
                ses.append("")
        lines.append(row(label, *coefs))
        lines.append(row("", *ses))

    lines.append(hline)

    # Additional indicator rows
    for lbl, vals in add_lines:
        lines.append(row(lbl, *vals))

    lines.append(hline)

    # Model statistics
    obs_vals = [str(m.n_obs) for m in models]
    ll_vals  = [f"{m.log_likelihood_:.3f}" for m in models]
    lines.append(row("Observations", *obs_vals))
    lines.append(row("Log Likelihood", *ll_vals))

    lines.append(hhline)
    ncol = n + 1
    note = (
        BS + f"multicolumn{{{ncol}}}{{l}}"
        + "{" + BS + "textit{Note:} "
        + "$^{*}$p$<$0.1; $^{**}$p$<$0.05; $^{***}$p$<$0.01}"
        + newrow
    )
    lines.append(note)
    lines.append(BS + "end{tabular}")
    lines.append(BS + "end{table}")

    tex = "\n".join(lines)
    with open(output_path, "w") as f:
        f.write(tex)
    print(f"Wrote {output_path}")
    return tex


tex = cox_to_latex(
    models        = [cox1, cox2, cox3],
    title         = "Risk of Rebellion --- Cox Proportional Hazards Model",
    display_covars= DISPLAY_COVARS,
    covar_labels  = COVAR_LABELS,
    column_labels = COLUMN_LABELS,
    add_lines     = ADD_LINES,
    output_path   = OUT_TEX,
)
print(tex)